# Enterprise Retail Intelligence Platform

## Notebook 06 : Business Insights

### Objective

Generate business KPIs and analytical insights from engineered retail datasets.

These insights will be used for:

- Executive Dashboard
- Sales Analysis
- Customer Analysis
- Product Analysis
- Transportation Analysis
- Return Analysis
- Power BI Dashboard

Author : Shobha Saxena

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path("../")

DATA_PATH = PROJECT_ROOT / "data" / "processed"

In [3]:
orders_df = pd.read_csv(
    DATA_PATH / "orders.csv",
    parse_dates=[
        "order_date",
        "ship_date",
        "created_at"
    ]
)

customers_df = pd.read_csv(DATA_PATH / "customers.csv")

products_df = pd.read_csv(DATA_PATH / "products.csv")

returns_df = pd.read_csv(
    DATA_PATH / "returns.csv",
    parse_dates=[
        "return_date",
        "created_at"
    ]
)

transportation_df = pd.read_csv(
    DATA_PATH / "transportation.csv",
    parse_dates=[
        "dispatch_date",
        "estimated_delivery_date",
        "actual_delivery_date",
        "created_at"
    ]
)

In [4]:
print("Orders:", orders_df.shape)
print("Customers:", customers_df.shape)
print("Products:", products_df.shape)
print("Returns:", returns_df.shape)
print("Transportation:", transportation_df.shape)

Orders: (9800, 14)
Customers: (793, 4)
Products: (1861, 5)
Returns: (490, 8)
Transportation: (9800, 10)


# Sales KPIs

In [5]:
# ============================================================
# Sales KPIs
# ============================================================

total_sales = orders_df["sales"].sum()

average_order_value = orders_df["sales"].mean()

highest_sale = orders_df["sales"].max()

lowest_sale = orders_df["sales"].min()

total_orders = orders_df["order_id"].nunique()

sales_summary = pd.DataFrame({
    "Metric": [
        "Total Sales",
        "Average Order Value",
        "Highest Sale",
        "Lowest Sale",
        "Total Orders"
    ],
    "Value": [
        round(total_sales, 2),
        round(average_order_value, 2),
        round(highest_sale, 2),
        round(lowest_sale, 2),
        total_orders
    ]
})

sales_summary

,Metric,Value
0,Total Sales,2261536.97
1,Average Order Value,230.77
2,Highest Sale,22638.48
3,Lowest Sale,0.44
4,Total Orders,4922.00


# Monthly Sales Trend

In [7]:
monthly_sales = (
    orders_df
    .groupby(
        orders_df["order_date"].dt.to_period("M")
    )["sales"]
    .sum()
    .reset_index()
)

monthly_sales["order_date"] = monthly_sales["order_date"].astype(str)

monthly_sales

,order_date,sales
0,2015-01,14205.71
1,2015-02,4519.92
2,2015-03,55205.83
3,2015-04,27906.86
4,2015-05,23644.30
5,2015-06,34322.94
6,2015-07,33781.52
7,2015-08,27117.53
8,2015-09,81623.52
9,2015-10,31453.37


# Regional Sales

In [8]:
regional_sales = (
    orders_df
    .groupby("region")["sales"]
    .agg(
        Total_Sales="sum",
        Average_Sales="mean",
        Orders="count"
    )
    .reset_index()
    .sort_values(
        by="Total_Sales",
        ascending=False
    )
)

regional_sales

,region,Total_Sales,Average_Sales,Orders
3,West,710219.77,226.184640,3140
1,East,669518.85,240.401741,2785
0,Central,492646.90,216.357883,2277
2,South,389151.45,243.524061,1598


# State Wise Sales

In [9]:
state_sales = (
    orders_df
    .groupby("state")["sales"]
    .sum()
    .reset_index()
    .sort_values(
        by="sales",
        ascending=False
    )
)

state_sales.head(10)

,state,sales
3,California,446306.49
30,New York,306361.07
41,Texas,168572.47
45,Washington,135206.87
36,Pennsylvania,116276.76
8,Florida,88436.55
11,Illinois,79236.57
20,Michigan,76136.07
33,Ohio,75130.43
44,Virginia,70636.72


# Customer KPIs

In [10]:
# ============================================================
# Customer KPIs
# ============================================================

total_customers = customers_df["customer_id"].nunique()

repeat_customers = (
    orders_df
    .groupby("customer_id")["order_id"]
    .nunique()
)

repeat_customers_count = (repeat_customers > 1).sum()

new_customers_count = total_customers - repeat_customers_count

customer_summary = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Repeat Customers",
        "New Customers"
    ],
    "Value": [
        total_customers,
        repeat_customers_count,
        new_customers_count
    ]
})

customer_summary

,Metric,Value
0,Total Customers,793
1,Repeat Customers,780
2,New Customers,13


# Top 10 Customers by Sales

In [11]:
top_customers = (
    orders_df
    .groupby("customer_id")["sales"]
    .sum()
    .reset_index()
    .sort_values(
        by="sales",
        ascending=False
    )
    .head(10)
)

top_customers

,customer_id,sales
700,SM-20320,25043.07
741,TC-20980,19052.22
621,RB-19360,15117.35
730,TA-21385,14595.62
6,AB-10105,14473.57
434,KL-16645,14175.23
669,SC-20095,14142.34
327,HL-15040,12873.30
683,SE-20110,12209.44
131,CC-12370,12129.08


# Customer Order Frequency

In [12]:
customer_frequency = (
    orders_df
    .groupby("customer_id")
    .agg(
        Total_Orders=("order_id", "nunique"),
        Total_Sales=("sales", "sum"),
        Average_Order_Value=("sales", "mean")
    )
    .reset_index()
)

customer_frequency.head()

,customer_id,Total_Orders,Total_Sales,Average_Order_Value
0,AA-10315,5,5563.56,505.778182
1,AA-10375,9,1056.39,70.426000
2,AA-10480,4,1790.51,149.209167
3,AA-10645,6,5086.94,282.607778
4,AB-10015,3,886.15,147.691667


# Customer Segment Summary

In [13]:
customer_sales = (
    orders_df
    .groupby("customer_id")["sales"]
    .sum()
)

high_threshold = customer_sales.quantile(0.75)
low_threshold = customer_sales.quantile(0.25)

customer_segment = customer_sales.reset_index()
customer_segment.columns = ["customer_id", "total_sales"]

customer_segment["customer_segment"] = np.where(
    customer_segment["total_sales"] >= high_threshold,
    "High Value",
    np.where(
        customer_segment["total_sales"] <= low_threshold,
        "Low Value",
        "Medium Value"
    )
)

customer_segment["customer_segment"].value_counts().reset_index()

,customer_segment,count
0,Medium Value,395
1,High Value,199
2,Low Value,199


# Product KPIs

In [14]:
# ============================================================
# Product KPIs
# ============================================================

total_products = products_df["product_id"].nunique()

product_summary = pd.DataFrame({
    "Metric": [
        "Total Products"
    ],
    "Value": [
        total_products
    ]
})

product_summary

,Metric,Value
0,Total Products,1861


# Top 10 Products by Sales

In [15]:
top_products = (
    orders_df
    .merge(
        products_df,
        on="product_id",
        how="left"
    )
    .groupby("product_name")["sales"]
    .sum()
    .reset_index()
    .sort_values(
        by="sales",
        ascending=False
    )
    .head(10)
)

top_products

,product_name,sales
398,Canon imageCLASS 2200 Advanced Copier,61599.83
639,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38
438,Cisco TelePresence System EX90 Videoconferenci...,22638.48
772,HON 5400 Series Task Chairs for Big and Tall,21870.57
673,GBC DocuBind TL300 Electric Binding System,19823.48
675,GBC Ibimaster 500 Manual ProClick Binding System,19024.50
791,Hewlett Packard LaserJet 3310 Copier,18839.68
773,HP Designjet T520 Inkjet Large Format Printer ...,18374.90
670,GBC DocuBind P400 Electric Binding System,17965.07
799,High Speed Automatic Electric Letter Opener,17030.31


# Top Categories by Sales

In [16]:
category_sales = (
    orders_df
    .merge(
        products_df,
        on="product_id",
        how="left"
    )
    .groupby("category")["sales"]
    .sum()
    .reset_index()
    .sort_values(
        by="sales",
        ascending=False
    )
)

category_sales

,category,sales
2,Technology,827455.94
0,Furniture,728658.75
1,Office Supplies,705422.28


# Top Sub-Categories by Sales

In [19]:
# ============================================================
# Top Sub-Categories by Sales
# ============================================================

subcategory_sales = (
    orders_df
    .merge(
        products_df,
        on="product_id",
        how="left"
    )
    .groupby("sub_category")["sales"]
    .sum()
    .reset_index()
    .sort_values(
        by="sales",
        ascending=False
    )
)

subcategory_sales.head(10)

,sub_category,sales
13,Phones,327782.49
5,Chairs,322822.75
14,Storage,219343.37
16,Tables,202810.77
3,Binders,200028.82
11,Machines,189238.68
0,Accessories,164186.70
6,Copiers,146248.07
4,Bookcases,113813.25
1,Appliances,104618.38


In [20]:
# ============================================================
# Product Performance
# ============================================================

product_performance = (
    orders_df
    .merge(
        products_df,
        on="product_id",
        how="left"
    )
    .groupby(["product_name", "category", "sub_category"])
    .agg(
        Total_Sales=("sales", "sum"),
        Orders=("order_id", "nunique"),
        Average_Sale=("sales", "mean")
    )
    .reset_index()
    .sort_values(
        by="Total_Sales",
        ascending=False
    )
)

product_performance.head(10)

,product_name,category,sub_category,Total_Sales,Orders,Average_Sale
398,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,61599.83,5,12319.966000
639,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,27453.38,10,2745.338000
438,Cisco TelePresence System EX90 Videoconferenci...,Technology,Machines,22638.48,1,22638.480000
772,HON 5400 Series Task Chairs for Big and Tall,Furniture,Chairs,21870.57,8,2733.821250
673,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,19823.48,11,1802.134545
675,GBC Ibimaster 500 Manual ProClick Binding System,Office Supplies,Binders,19024.50,9,2113.833333
791,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,18839.68,8,2354.960000
773,HP Designjet T520 Inkjet Large Format Printer ...,Technology,Machines,18374.90,3,6124.966667
670,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,17965.07,6,2994.178333
799,High Speed Automatic Electric Letter Opener,Office Supplies,Supplies,17030.31,3,5676.770000


# Transportation KPIs

In [21]:
# ============================================================
# Transportation KPIs
# ============================================================

transportation_df["delivery_days"] = (
    transportation_df["actual_delivery_date"] -
    transportation_df["dispatch_date"]
).dt.days

transportation_summary = pd.DataFrame({
    "Metric": [
        "Average Delivery Days",
        "Maximum Delivery Days",
        "Minimum Delivery Days",
        "Average Delivery Cost"
    ],
    "Value": [
        round(transportation_df["delivery_days"].mean(), 2),
        transportation_df["delivery_days"].max(),
        transportation_df["delivery_days"].min(),
        round(transportation_df["delivery_cost"].mean(), 2)
    ]
})

transportation_summary

,Metric,Value
0,Average Delivery Days,4.11
1,Maximum Delivery Days,7.00
2,Minimum Delivery Days,3.00
3,Average Delivery Cost,29.94


# Carrier Performance

In [22]:
carrier_performance = (
    transportation_df
    .groupby("carrier_name")
    .agg(
        Shipments=("transport_id", "count"),
        Average_Delivery_Days=("delivery_days", "mean"),
        Average_Delivery_Cost=("delivery_cost", "mean")
    )
    .reset_index()
    .sort_values(
        by="Shipments",
        ascending=False
    )
)

carrier_performance

,carrier_name,Shipments,Average_Delivery_Days,Average_Delivery_Cost
5,Xpressbees,1676,4.086436,30.024994
1,Delhivery,1658,4.107733,30.009729
2,Dhl,1636,4.123641,29.634688
3,Fedex,1635,4.093579,29.868532
4,Ups,1626,4.127820,30.060873
0,Blue Dart,1569,4.110412,30.029726


# Return KPIs

In [23]:
# ============================================================
# Return KPIs
# ============================================================

total_returns = returns_df["return_id"].nunique()

total_refund = returns_df["refund_amount"].sum()

return_summary = pd.DataFrame({
    "Metric": [
        "Total Returns",
        "Total Refund Amount"
    ],
    "Value": [
        total_returns,
        round(total_refund, 2)
    ]
})

return_summary

,Metric,Value
0,Total Returns,490.00
1,Total Refund Amount,81807.14


# Return Reasons

In [24]:
return_reason_summary = (
    returns_df
    .groupby("return_reason")
    .agg(
        Returns=("return_id", "count"),
        Refund=("refund_amount", "sum")
    )
    .reset_index()
    .sort_values(
        by="Returns",
        ascending=False
    )
)

return_reason_summary

,return_reason,Returns,Refund
3,Late Delivery,105,14606.05
2,Defective Product,92,10902.07
0,Customer Changed Mind,85,15561.82
5,Wrong Item,72,16337.42
1,Damaged Product,70,10474.53
4,Other,66,13925.25


# Return Status Summary

In [25]:
return_status_summary = (
    returns_df
    .groupby("return_status")
    .agg(
        Returns=("return_id", "count")
    )
    .reset_index()
)

return_status_summary

,return_status,Returns
0,Approved,42
1,Completed,338
2,Rejected,65
3,Requested,45


# Executive Dashboard Summary

In [26]:
# ============================================================
# Executive KPI Summary
# ============================================================

executive_summary = pd.DataFrame({

    "KPI":[
        "Total Sales",
        "Total Orders",
        "Total Customers",
        "Total Products",
        "Average Order Value",
        "Average Delivery Days",
        "Total Returns",
        "Total Refund Amount"
    ],

    "Value":[
        round(orders_df["sales"].sum(),2),
        orders_df["order_id"].nunique(),
        customers_df["customer_id"].nunique(),
        products_df["product_id"].nunique(),
        round(orders_df["sales"].mean(),2),
        round(transportation_df["delivery_days"].mean(),2),
        returns_df["return_id"].nunique(),
        round(returns_df["refund_amount"].sum(),2)
    ]

})

executive_summary

,KPI,Value
0,Total Sales,2261536.97
1,Total Orders,4922.00
2,Total Customers,793.00
3,Total Products,1861.00
4,Average Order Value,230.77
5,Average Delivery Days,4.11
6,Total Returns,490.00
7,Total Refund Amount,81807.14


# Data Quality Summary

In [27]:
# ============================================================
# Data Quality Summary
# ============================================================

quality_summary = pd.DataFrame({

    "Dataset":[
        "Orders",
        "Customers",
        "Products",
        "Transportation",
        "Returns"
    ],

    "Rows":[
        len(orders_df),
        len(customers_df),
        len(products_df),
        len(transportation_df),
        len(returns_df)
    ],

    "Columns":[
        orders_df.shape[1],
        customers_df.shape[1],
        products_df.shape[1],
        transportation_df.shape[1],
        returns_df.shape[1]
    ]

})

quality_summary

,Dataset,Rows,Columns
0,Orders,9800,14
1,Customers,793,4
2,Products,1861,5
3,Transportation,9800,11
4,Returns,490,8


# Notebook Summary

## Business Insights Generated

### Sales Insights
- Total Sales
- Average Order Value
- Monthly Sales Trend
- Regional Sales
- State Sales

### Customer Insights
- Customer Summary
- Customer Frequency
- Top Customers
- Customer Segmentation

### Product Insights
- Product Summary
- Category Sales
- Sub Category Sales
- Product Performance

### Transportation Insights
- Carrier Performance
- Delivery Performance

### Return Insights
- Return Summary
- Return Reasons
- Return Status

### Executive Summary
- Dashboard KPIs
- Data Quality Summary

Notebook Status

✅ Completed Successfully